Para emepezar, poner el jupyter en vs code con el kernel de python

Si no os sale, ejecutad estos tres comandos a la altura del path T20
- python3 -m venv .venv
- source .venv/bin/activate
- python -m pip install ipykernel

Sección	Contenido	Responsable
0. Introducción	Objetivo, integrantes y librerías	JORGE
1. Reparto	Apartado a): generación de participaciones	JORGE
2. Recuperación	Apartado b): interpolación de Newton	ARJO
3. Comprobación	Apartado c): ejemplo del profesor	Compañero INIESTA
4. Umbral	Apartado d): pruebas y explicación matemática	Compañero INIESTA
5. Aplicación AES	Apartado e): clave AES-256	FRAN
6. Lagrange	Apartado f): recuperación alternativa	FRAN (OPCIONAL)
7. Conclusiones	Resultados y comprobaciones finales	TODOS

Cada sección tendrá una explicación breve, su código Python comentado y los resultados de ejecución.
-
-
-
-
-
-
-
-
-
-
-
-


# Seguridad y Protección de Sistemas Informáticos
## Trabajo de Python — Esquema de Shamir

**Curso:** 2026/2027

**Integrantes:**
- Jorge Castillo Fuentes:
- Alfredo Iniesta Garcia:
- Francisco Pradillo Martinez:
- Alvaro Arjona Jimenez:

## Objetivo

Implementar el esquema de Shamir para compartir secretos,
permitiendo repartir un secreto entre varios participantes
y recuperarlo cuando se dispone de suficientes participaciones.

## Apartados

1. Reparto del secreto
2. Recuperación mediante interpolación de Newton
3. Comprobación con el ejemplo de los apuntes
4. Comprobación del umbral de seguridad
5. Aplicación con una clave AES de 256 bits
6. Interpolación de Lagrange (opcional)

## 1. Reparto del secreto

En este apartado se implementará una función que permita
dividir un secreto en n participaciones mediante el
esquema de Shamir.

Para recuperar el secreto se necesitarán al menos
t participaciones.

La función recibirá:
- s: secreto que queremos repartir.
- n: número de participantes.
- t: número mínimo de participaciones.
- bits: tamaño máximo del secreto en bits.

La función devolverá el número primo p y las
participaciones generadas.

In [13]:
import secrets
import subprocess
from itertools import combinations
import time
import random

## 1. Reparto del secreto

En este apartado implementamos el reparto de un secreto
mediante el esquema de Shamir.

La función recibe el secreto, el número de participantes,
el umbral mínimo para recuperarlo y el tamaño máximo
del secreto en bits.

Primero comprobamos los parámetros. Después generamos
un número primo y los coeficientes aleatorios del
polinomio. Por último, elegimos los puntos y calculamos
las participaciones de cada participante.

Todas las operaciones del polinomio se realizan módulo p.

In [14]:
def validar_parametros(s, n, t, bits):
    # Comprobamos las condiciones del enunciado.
    if not all(type(v) is int for v in (s, n, t, bits)):
        raise TypeError("Los parámetros deben ser enteros")

    if n < 3:
        raise ValueError("Debe haber al menos 3 participantes")

    if t < 2 or t > n:
        raise ValueError("El umbral debe estar entre 2 y n")

    if bits < 1:
        raise ValueError("El tamaño en bits debe ser positivo")

    if s < 0 or s >= 2**bits:
        raise ValueError("El secreto no cabe en los bits indicados")


def generar_primo(bits, n):
    # El primo debe superar tanto a n como a cualquier
    # secreto que pueda representarse con esos bits.
    bits_primo = max(bits + 1, n.bit_length() + 1)

    resultado = subprocess.run(
        ["openssl", "prime", "-generate", "-bits", str(bits_primo)],
        capture_output=True,
        text=True,
        check=True
    )

    return int(resultado.stdout.strip())


def evaluar_polinomio(coeficientes, x, p):
    # Evaluamos el polinomio módulo p mediante Horner.
    resultado = 0

    for coeficiente in reversed(coeficientes):
        resultado = (resultado * x + coeficiente) % p

    return resultado


def repartir_secreto(s, n, t, bits):
    validar_parametros(s, n, t, bits)

    # Generamos el primo.
    p = generar_primo(bits, n)

    # El primer coeficiente es el secreto.
    # Los demás se generan aleatoriamente, incluido el cero.
    coeficientes = [s]
    for _ in range(t - 1):
        coeficientes.append(secrets.randbelow(p))

    # Elegimos n puntos diferentes y distintos de cero.
    puntos = set()

    while len(puntos) < n:
        x = secrets.randbelow(p - 1) + 1
        puntos.add(x)

    # Calculamos la participación correspondiente a cada x.
    participaciones = []

    for x in sorted(puntos):
        y = evaluar_polinomio(coeficientes, x, p)
        participaciones.append((x, y))

    return p, participaciones

In [15]:
# Ejemplo con 5 participantes y umbral de 3.
s = 1234
n = 5
t = 3
bits = 16

p, participaciones = repartir_secreto(s, n, t, bits)

print("Secreto:", s)
print("Número primo:", p)
print("Participantes:", n)
print("Umbral:", t)

print("\nParticipaciones generadas:")
for i, (x, y) in enumerate(participaciones, start=1):
    print(f"Participante {i}: ({x}, {y})")

print("\nComprobaciones:")
print("Primo mayor que n:", p > n)
print("Primo mayor que todos los secretos posibles:", p >= 2**bits)
print("Número correcto de participaciones:", len(participaciones) == n)
print("Puntos distintos:", len({x for x, y in participaciones}) == n)
print("Puntos no nulos:", all(x != 0 for x, y in participaciones))

Secreto: 1234
Número primo: 127249
Participantes: 5
Umbral: 3

Participaciones generadas:
Participante 1: (18874, 32157)
Participante 2: (26410, 102309)
Participante 3: (29558, 82168)
Participante 4: (78664, 33341)
Participante 5: (96687, 87433)

Comprobaciones:
Primo mayor que n: True
Primo mayor que todos los secretos posibles: True
Número correcto de participaciones: True
Puntos distintos: True
Puntos no nulos: True


# 5. e) Uso realista. Aplicación de una clave AES de 256 bits.
En este apartado aplicaremos las funciones y conclusiones obtenidas tanto de los apuntes como de los apartados anteriores para aplicar el esquema de Shamir a un caso realista. El objetivo es repartir una clave AES de 256 bits entre 5 participantes, siendo necesarias 3 participaciones para recuperar el valor original de la clave. 
En consecuencia, nuestros parámetros serán $n=5$ y $t=3$. 
Para realizar la repartición del secreto y su recuperación, utilizaremos las funciones previamente creadas en los apartados anteriores. Dichas funciones trabajan con números enteros, por tanto deberemos convertir nuestra clave a un número entero $s$ antes de realizar el reparto. Necesitaremos, además, un número primo $p$ de 257 bits tal que $p>s$.
Finalmente, se recuperará el secreto utilizando 3 de las 5 participaciones, convertiremos dicho secreto nuevamente a bytes y comprobaremos que coincide con el secreto original. 
Durante esta demostración se mostrarán parámetros (como claves o participaciones) que no se mostrarían en un caso real, puesto que sería confidencial. En este caso dicha información se mostrará por pantalla para demostrar la funcionalidad del caso. 
A continuación se muestra la resolución del problema:


In [16]:
# Establecemos los parámetros de nuestro problema
n = 5
t = 3
bits = 256

# Generamos una clave AES de 256 bits (32 bytes)
clave_AES = secrets.token_bytes(32)

# Visualizamos la clave generada en hexadecimal para posteriormente poder comprobarla
print("Clave secreta original:", clave_AES.hex())

# Convertimos nuestra clave a un número entero
s = int.from_bytes(clave_AES, "big")
print("Clave secreta original en entero:", s)

# Repartimos el secreto en base a nuestras variables
# Para dicha repartición utilizaremos la función repartir_secreto(s, n, t, bits) del apartado a)
# En ella ya se genera el número p de 257 bits que necesitamos
p, participaciones = repartir_secreto(s, n, t, bits)

# Mostramos tanto el número p generado, como las participaciones generadas para cada participante
print("Número primo (p):", p)
print ("\nParticipaciones:")
for i, (x, y) in enumerate(participaciones, start=1):
    print(f"Participante {i}: ({x}, {y})")
    
# Seleccionamos t=3 de las participaciones obtenidas de forma aleatoria sin repetir ninguna para la recuperación de la clave
participaciones_seleccionadas = random.sample(participaciones, t)

# Muestro las participaciones seleccionadas
print("\nParticipaciones seleccionadas para recuperar la clave: ")
for x, y in participaciones_seleccionadas:
    print(f"({x}, {y})")

# Comenzamos la recuperación del secreto utilizando la función de interpolación de Newton correspondiente al apartado b)


# Convertimos el secreto recuperado a su formato original de 32 bytes
clave_AES_recuperada = s_recuperado.to_bytes(32, "big")

# Mostramos la clave original y la recuperada en formato hexadecimal para comparar
print("\nClave AES original: ", clave_AES.hex())
print("\nClave AES recuperada: ", clave_AES_recuperada.hex())

# Compruebo que ambas coinciden
if clave_AES==clave_AES_recuperada:
    print("\nLa clave original se ha recuperado.")
else:
    print("\nLa clave original no se ha podido recuperar.")

Clave secreta original: 7fe6087558e7fbca5dc43da86dcd85640e2a03ef28defe60a907f11bd7bc48e9
Clave secreta original en entero: 57850164972621308267092623360584969333984573636146745963084316948750642465001
Número primo (p): 221493326742458794708585685432918055986223190347060645760198280461039735054387

Participaciones:
Participante 1: (39551745260517970445898555593431843592777749898886558831640387830441323559741, 197922348664176702708239888594829200894780733873102025123713298226243104551696)
Participante 2: (41226568425639443340007880760444518926929828479014937619662847135720022633505, 115963220800812184073135313090717048952042208047149648820291550768163661955580)
Participante 3: (165837354023377604596870591454046482477293658019693709164144200632957631502883, 46746956346983063780128384306145539229811438054047287652128185888573700577972)
Participante 4: (175255613938072889843607508174634131258270871870798390201995534500818829983288, 20695164027302015411471404535767847759038217048648656278996

NameError: name 's_recuperado' is not defined

# 6. f) Recuperación mediante la interpolación de Lagrange en 0.
Finalmente, implementaremos otro método para recuperar el secreto en base a la interpolación de Lagrange. Además se comparará este método en base a sus resultados y tiempos con el método basado en la interpolación de Newton.
Adaptando las fórmulas del teorema de Lagrange a las participaciones del esquema de Shamir denotadas como $t$, obtenemos las siguientes fórmulas:
$$
P(x)=\sum_{i=1}^{t}y_iL_i(x)
$$

donde: 

$$
L_i(x)=\prod_{\substack{j=1\\j\neq i}}^{t}\frac{x-x_j}{x_i-x_j}
$$

Para recuperar el valor del secreto ($s$), el cual es el término independiente, únicamente debemos dar el valor $x=0$ para obtenerlo.

Para implementar dicha fórmula se utilizarán dos bucles. En uno recorreremos cada participación, en el segundo calcularemos el producto correspondiente en función del resto de participaciones.
Cada operación se realizará en módulo p (reduciremos con %), para ello usaremos pow() para calcular los inversos necesarios.

En base a estas consideraciones resulta la siguiente función recuperar_lagrange(p, participaciones)

In [17]:
def recuperar_lagrange(p, participaciones):
    # Establecemos los parametros t = número de participaciones y s = 0, correspondiente al secreto que queremos recuperar
    t = len(participaciones)
    s = 0
    
    # Recorremos cada participación guardando sus valores (x, y) de cada una de ellas
    for i in range(t):
        x_i, y_i = participaciones[i]
        
        # Inicializo el producto L_i en 1
        L_i = 1
        
        # Recorremos el resto de participaciones guardando la coordenada x_j de cada una de ellas para calcular el producto L_i
        for j in range(t):
            if j != i:
                x_j = participaciones[j][0]
                
                # Establezco el numerador y el denomidador de la operacion L_i en base a la fórmula
                numerador = (-x_j)
                denominador =  (x_i - x_j) % p
                
                # Calculamos el inverso en Zp del denominador para poder realizar la división
                inverso_denominador = pow(denominador, -1, p)
                
                # Calculamos el producto L_i en módulo p 
                L_i = (L_i * numerador * inverso_denominador) % p
                
        # Añado el resultado a s de cada producto multiplicado por y_i siguiendo la fórmula de P s=P(0)=Σy_i*L_i
        s = (s + y_i * L_i) % p
                
    # Devolvemos el secreto recuperado
    return s
                

Una vez tenemos la función, compararemos los resultados y los tiempos con el método basado en la interpolación de Newton. Usaremos para ello la librería time, con su función time.perf_counter():

In [ ]:
# Medimos el tiempo de recuperación con el método de Lagrange
inicio = time.perf_counter()

s_lagrange = recuperar_lagrange(p, participaciones_seleccionadas)

fin = time.perf_counter()

tiempo = fin - inicio

print("\nSecreto recuperado con el método de Lagrange:", s_lagrange)
print("\nTiempo de recuperación con el método de Lagrange:", tiempo, "segundos")